#Constraint Satisfaction Problem
Problem definition:
1. A finite set of variables and their domains.
2. A finite set of conditions on these variables.
3. A solution is an assignment to these variables that satisfies all the conditions.

##Example: 8-Queens Puzzle
The 8-queens puzzle is the problem of placing eight chess queens on an $8×8$ chessboard so that **no two queens threaten each other**; thus, a solution requires that **no two queens share the same row, column, or diagonal**.

Variables: $q_1,..., q_8$, the position of a queen in column 1,...,8.

Domain: the same for all variables, {1,..,8}.
Constraints:
 1. $q_i −q_j \neq 0,$ for $i \neq j \quad i ∈ {1,2,...,8} \quad j ∈ {1,2,...,8}$
 2. $|q_{i} - q_{i+k}| \neq k$ for $ k = 1,2,...,7\quad i = 1,...,8 −k$

###Constructive Methods
**Constructive methods use search strategies, especially depth-first search**, to try to find a solution:
1. States: any partial assignment (assign values to some of the variables).
2. Initial state: the empty assignment.
3. Operator: **pick a new variable, and assign a value in its domain to it**.
4. Path cost: 0.
5. Goal condition: when all constraints are satisfied.

What is the search space? (include the illegal one)
$$\Omega =  \{\emptyset, 1, 2, ..., 8\}^{8} $$
但是这种使用Depth-First Search的方法太费时间了，我们想要一个更好的算法。

###Constraint Propagation
A technique called **constraint propagation** can be used to **prune the search space**: given the current partial assignment, one can use the constraints to narrow down the search, i.e. eliminate some possible values of the remaining variables.

例子：
| Round | $q_1$ | $q_2$ | $q_3$ | $q_4$ |
| :---  | :---  | :---  | :---  | :---  |
| 0     | $\{1, 2, 3, 4\}$ | $\{1, 2, 3, 4\}$ | $\{1, 2, 3, 4\}$ | $\{1, 2, 3, 4\}$ |
| 1     | $\{1\}$ | $\{3, 4\}$ | $\{2, 4\}$ | $\{2, 3\}$ |
| 2     | $\{1\}$ | $\{4\}$ | $\{2\}$ | $\{2\}$ |
| 3     | $\{1\}$ | $\{4\}$ | $\{2\}$ | $\{\emptyset\}$ |

#Satisfiability (SAT):
**Satisfiability (SAT)**: the problem of deciding if a CNF (or a set of clauses) is satisfiable.

**Literal**: a variable or the negation of a variable.

**Clause**: a disjunction of literals. E.g., $C = l_1 ∨ l_2$

**Conjunctive Normal Form (CNF)**: a conjunction of one or more clauses, can be equivalently represented using a set of clauses. E.g., $\{C_1,C_2\}$, equivalent with $C_1 ∧ C_2$

**3SAT**: the problem of deciding if a set of clauses that have **no more than 3 literals is satisfiable**. 我们可以证明3SAT和其他SAT是等价的，所以研究者一般就只研究3SAT

##Procedure for SAT
A **procedure for SAT** is a procedure to **check whether the input is satisfiable or not**. (procedure是可以用于检测我们的input是否满足我们的条件)

输入：a set of Clause: $\{C_1, C_2, ...\}$ 或者 CNF: $C_{1} \wedge C_{2} \wedge ...$

输出：$Yes$或$No$

我们有三类procedure:
1. Sound procedure: 当它返回$Yes$时，说明我们的input一定是satisfiable的；但是对有些satisfiable的input，Sound procedure会返回$No$
2. Complete procedure: 如果它返回$No$, 那么他一定是unsatistfied input; 如果它返回$Yes$, 那我们的input是有可能是satisfiable的
3. Incomplete procedure:  the procedure may return $No$ for **some satisfiable input**; the procedure may return $Yes$ for **some unsatifiable input**. （所有procedure都可以是Incomplete procedure, 所以我们一般都只讨论incomplete and sound procedure）

##DPLL Procedure
The most **popular and widely studied complete method for SAT is the so called Davis-Putnam-Longemann-Loveland (DPLL) procedure**.

算法目标：通过系统的探索变量(variable), 来判定一个CNF是否存在使其为True的一组赋值。如果存在，返回True; 如果不存在，返回False.

输入：$$\alpha = \{C_{1}, C_{2}, ...\}, \text{where } C_{i} \text{ is the disjunction of variable }l_{i_{1}}, l_{i_{1}}, ...$$
输出：$True$ or $False$

Let $α$ be a **set of clauses**: $\{C_{1}, C_{2}, ...\}$. If $l$ is a literal occurring in $α$ ($l$ is a variable or the negation of a variable), then by $α(l)$, we mean the result
 of letting $l$ be true. $α(l)$ is a procedure:
 如果$C$中包含$l$，则从$C$中删除这个Clauses; 如果$C$中包含$\neg l$，则从$C$中删除这个Clauses 如果$C$中不包含$l$或者$\neg l$，则从$C$中删除这个Clauses.

``` python
α(l):
  Let l be True:
  for ∀C ∈ α:
    if C mention l:
      eliminate C in α
    else if C mention ¬l:
      delete ¬l from C
```
``` python
Procedure DPLL(α):
  if α is empty:
    then return yes.
  else if there is an empty clause in α:
    return no.
  else if there is a pure literal l in α:
    then return DPLL(α(l)).
  else if there is a unit clause {l} in α:
    then return DPLL(α(l)).
  else select a variable p in α, and do the following
    if DPLL(α(p)) return yes:
      then return yes.
    else:
      return DPLL(α(¬p)).
```

这个DPLL procedure是怎么实现的？
1. 如果$\alpha$是空的，说明没有限制条件，返回$True$
2. 如果一个Clause只有一个variable (unit Clause), 直接将这个variable赋值为$True$, 而后使用$\alpha(l)$将其化简
3. 如果在所有Clause中，variable $l$都以$l$或$\neg l$形式出现 (pure literal), 直接将这个variable赋值为$True$, 而后使用$\alpha(l)$将其化简
4. 如果出现$C_{i} = \{\} = \{False \vee False\}$ (empty Clause), 直接返回$False$
5. 选取一个variable $l$ in $\alpha$, 如果$DPLL(\alpha(l))$是$True$, 直接返回$True$即可; 否则返回$DPLL(\alpha(\neg l))$

###Example
let: $$\alpha = \{\neg a \vee d, a \vee \neg b \vee c, \neg c\}$$

steps:
1. unit case: $\neg c$, we can let $c = False$, and do $\alpha(\neg c)$, then our $\alpha$ becomes: $\{\neg a \vee d, a \vee \neg b\}$
2. pure literal: $d$ and $\neg b$, we can let $d = True$ and $b = False$, then our $\alpha$ becomes: $\{\neg a, a\}$
3. 选取$l = a$, calculate $DPLL(\alpha(a))$ and $DPLL(\alpha(\neg a))$. We find that both of them return $False$
4. end the algorithm and return $False$

##GSAP (an example for incomplete method)
Incomplete methods cannot be used to check unsatisfiable CNFs. But on satisfiable CNFs, incomplete methods can sometimes be much faster than the currently known best complete methods.

The following randomized local search algorithm, often called GSAT, is typical of current best known incomplete methods:

```python
Procedure GSAT(α, max-restart: mr, max-climb: mc):
  for i = 1 to mr:
    get a randomly generated truth assignment v.
    for j = 1 to mc:
      if v satisfies α:
        return yes
      else
        let v be a random choice of one best successors of v
  return failure
```
Here, **mr** is: max restart times (最大重启次数); **mc** is: max climbing times (最大探索次数，表示在一次启动中会在局部搜索多少次)

注意到：如果我们的程序如果返回了值$v$, 那么$v$ must satisfy $\alpha$, 所以GSAT一定是Sound Procedure.
###Example of GSAP:
$α = \{C_1,C_2,C_3,C_4\} = \{a∨¬b∨c∨¬d,¬a∨c,¬c∨d,a\}$

Variables: $a,b,c,d$. Domain: $\{True, False\}$.
```
在round-1的时候 (i = 1 < mr = 10):
  随机给variable (a, b, c, d)赋值：{a = False,b = True,c = True,d = True}.
  j = 1 (first round of climb):
    v 不满足 α, 因为 C4 没有被满足.
    Generate all successors of v by flipping one varible of v:
    v1 = {a = True,b = True,c = True,d = True}, # satisfied clauses = 4.
    v2 = {a = False,b = False,c = True,d = True}, # satisfied clauses = 3.
    v3 = {a = False,b = True,c = False,d = True}, # satisfied clauses = 2.
    v4 = {a = False,b = True,c = True,d = False}, # satisfied clauses = 2.
    Among all the successors, v1 is the best. Let v = v1.
  j = 2 (second round of climb):
    v satisfies α: return yes.

```

##Heuristic Repair
GSAT is a local search method that belongs to Heuristic repair class of methods in CSP.

In CSP, constructive methods starts with empty assignment, and build up a solution gradually by assigning values to variables one by one.

Heuristic repair starts with a proposed solution, which most probably does not satisfy all constraints,and repair it until it does.

###Min-Conflict
原理：每一次都选择和约束条件冲突最小的那一个state去expand
例子：

第一次：$$\text{# conflicts} = (1, 0, 3, 4)$$选择conflict为0的那个state，然后expand它

第二次：$$\text{# conflicts} = (2, 1, 1, 3)$$选择conflict为1的那个state (随机挑一个)，然后expand它

###K-Means Clustering
我们可以将K-Mean Clustering变成CSP (Constraint Search Problem)的形式：

Variables: $c_1,...,c_n, \mu_1,...,\mu_K$, where $\mu_{i}$ represents for centroids

Domains: $\{1,...,K\}$ for $c_i$ and the feature space for $\mu_i$ (i.e. the domain for $ϕ(x)$).

Constraints: minimize the following:
$$\sum_{i=1}^{n} ||ϕ(x_i) − \mu_{c_{i}} ||^{2}$$

方法：
1. Initialize $\mu_1,...,\mu_K$.
2. Iterate the following until termination condition:
- Compute the best assignment $c_1,...,c_n$ for the given $\mu$.
- Computer the best centroids $\mu_1,...,\mu_K$ for the given $c$.

##Hill-Climbing Search
Basic ideas:
1. Start at the initial state.
2. At each step, **move to a next state with highest value**.
3. It does not maintain the search tree and does not backtrack- it kees only the current node and its evaluation.

所以，Hill-Climbing Search是一种Greedy algorithm

Hill-climbing can **get easily stuck in a local maximum**.

The main idea:
1. At each step, instead of picking the best move, it **picks a random move**.
2. If the move leads to a state with **higher value**, then execute the move.
3. Otherwise, execute the **move with certain probability $p$ that becomes smaller as the algorithm progresses and with certain probability $(1 - p)$ stay in the current state.**
4. The probability is computed according to a function (schedule) that maps time to probabilities.

The idea comes from the process of **annealing- cooling metal liquid gradually until it freezes**.